# Week 13 Lab · LLMs: One Model, Different Decoding Rules
**심층생성모델 (GDS0058-1) · 13주차 · 강의자료 v16 대응**  
2026-11-25 · 학생용 실습 노트북 v4

지난 시간에는 고정된 Diffusion 모델에서 CFG의 $w$를 바꾸었다. 이번에는 **고정된 AR 모델의 다음 토큰 확률**에서 decoding rule을 바꿔 본다. 확률이 높은 답, 다양한 답, 과업에 맞는 답이 언제나 같은 것은 아니다.

**오늘의 목표:** ① 온도와 후보 제한이 분포를 어떻게 바꾸는지 예측한다. ② 같은 난수로 비교한다. ③ 과업의 요구사항에 맞는 규칙을 고른다. ④ temperature softmax를 한 줄 완성한다.

| 본실습 (24분) | 하는 일 | v16 슬라이드 |
|---|---|---|
| §0 (1분) | 비교 조건 및 재현성 고정 | 11, 24–25 |
| §1 (4분) | 코드 실행 전 결과 예측 | 17–19 |
| §2 (8분) | 200회 추출, 이론/관측 비교, 문맥 C | 17–20 |
| §3 (5분) | 일관성·다양성·허용 규칙 판단 | 19–21 |
| §4 (6분) | temperature softmax TODO + 자가검증 | 17, 25 |

**실행 방법:** §0–§4 순서대로 실행. 본실습은 Python 표준 라이브러리만 사용하며 GPU, API key, 모델 다운로드가 필요하지 않다. §5–§7은 시간 여유가 있을 때 하는 복습이고, §8의 실제 pretrained 모델만 인터넷과 추가 라이브러리가 필요하다. §1과 §3의 서술 답안은 스스로 작성한다.

> 이 노트북은 실제 LLM의 학습이나 성능 평가가 아닌, 강의의 가상 확률로 decoding을 분리해 검증하는 실험이다. 한 위치에서 반복 추출한 토큰 200개는 200토큰짜리 문장 하나가 아니다.

## §0. Setup — 실험 조건 고정하기 (1분)

**슬라이드 11**의 `The cat sat on the` 문맥을 사용한다. 확률은 이미 학습된 모델이 냈다고 가정한다. 세 문맥 모두 어휘는 같고 확률만 다르다.

- **Context A:** 원래 확률. **B:** 첫 토큰에 집중된 확률. **C:** A에서 `mat`과 `roof`의 확률만 교환.
- **Allowed labels:** 과업에서 허용하는 `mat`, `floor`, `couch`. `roof`, `moon`은 과업 규칙상 금지이며 언어 자체가 틀렸다는 뜻은 아니다.
- **Matched draws:** 모든 확률적 규칙에 동일한 난수 200개(seed=13)를 재사용한다.

In [ ]:
from collections import Counter
from math import exp, isclose, log
from random import Random

PROMPT = "The cat sat on the"
TOKENS = ("mat", "floor", "couch", "roof", "moon")
PROBS = (0.62, 0.21, 0.09, 0.05, 0.03)  # Context A
CONCENTRATED = (0.94, 0.03, 0.015, 0.01, 0.005)  # Context B
CONTEXT_C = (0.05, 0.21, 0.09, 0.62, 0.03)
CONTEXTS = {"A": PROBS, "B": CONCENTRATED, "C": CONTEXT_C}
ALLOWED = {"mat", "floor", "couch"}
LOGITS = tuple(log(p) for p in PROBS)  # equivalent toy logits at T=1
SEED, N_DRAWS = 13, 200
rng = Random(SEED)
draws = [rng.random() for _ in range(N_DRAWS)]

assert all(isclose(sum(p), 1.0) and all(q >= 0 for q in p) for p in CONTEXTS.values())
print(f"Prompt: {PROMPT} ___ | seed={SEED}, draws={N_DRAWS}")
print(f"{'Token':<10}{'A':>9}{'B':>9}{'C':>9}  Task")
for i, token in enumerate(TOKENS):
    print(f"{token:<10}{PROBS[i]:>9.3f}{CONCENTRATED[i]:>9.3f}{CONTEXT_C[i]:>9.3f}  "
          f"{'allowed' if token in ALLOWED else 'forbidden'}")

## §1. Predict — 실행 전에 결과 예측하기 (4분)

슬라이드 17–19를 떠올리자. Context A의 최고 확률은 `mat`의 0.62이고, `roof`와 `moon`의 확률 합은 0.08이다.

1. Greedy로 같은 문맥에서 200번 뽑으면 어떤 토큰을 몇 번 반환할까?
2. $T=1$에서 $T=2$로 높이면 낮은 확률의 후보가 뽑힐 가능성은 어떻게 달라질까? 허용되지 않은 토큰의 확률은?
3. Context A와 B에 각각 top-k=2와 top-p=0.90을 적용하면, 몇 개의 후보가 남을까?

**내 예측 (셀을 편집하여 작성)**  
Q1: …  
Q2: …  
Q3: …

## §2. Run & Compare — 같은 난수로 비교하기 (8분)

### 2-1. 확률을 바꾼 뒤 후보를 남긴다 (슬라이드 17–18)

모델의 logit을 $z_i$, 원래 확률을 $p_i$라고 하자. 양수인 온도 $T$를 적용하면

$$p_i(T)=\frac{\exp(z_i/T)}{\sum_j\exp(z_j/T)}=\frac{p_i^{1/T}}{\sum_j p_j^{1/T}}.$$

$T=0.5$는 제곱, $T=2$는 제곱근으로 확인할 수 있다. 온도는 확률을 재가중하고, top-k/top-p는 후보를 제거한다. **Top-p는 누적합이 처음 문턱 이상이 되는 토큰까지 포함한다.** 제거한 토큰의 확률은 0, 남은 확률은 다시 정규화한다. 이 실습에서는 한 번에 한 종류의 cutoff만 사용한다.

아래 준비된 함수를 실행한다. 모든 확률은 강의와 같은 가상 값이다. 실험 중 모델 가중치를 학습하지 않는다.

In [ ]:
def candidate_weights(probabilities, *, top_k=None, top_p=None):
    # Sort by descending mass; stable ordering breaks ties by token index.
    if top_k is not None and not 1 <= top_k <= len(probabilities):
        raise ValueError("top_k must be within the vocabulary size.")
    if top_p is not None and not 0 < top_p <= 1:
        raise ValueError("top_p must be within (0, 1].")
    if top_k is not None and top_p is not None:
        raise ValueError("Use one cutoff at a time in this lab.")
    ranked = sorted(range(len(probabilities)), key=lambda i: -probabilities[i])
    selected = ranked[:top_k]
    if top_p is not None:
        selected, cumulative = [], 0.0
        for index in ranked:
            selected.append(index)
            cumulative += probabilities[index]
            if cumulative >= top_p:
                break
    total = sum(probabilities[i] for i in selected)
    if total <= 0:
        raise ValueError("No positive probability remains.")
    return [(i, probabilities[i] / total) for i in selected if probabilities[i] > 0]


def apply_rule(probabilities, *, temperature=1.0, top_k=None, top_p=None):
    if temperature <= 0:
        raise ValueError("Temperature must be positive; use greedy separately.")
    if not probabilities or min(probabilities) < 0 or max(probabilities) <= 0:
        raise ValueError("Use nonnegative weights with at least one positive value.")
    # Equivalent to softmax(logit / T); scaling by peak avoids overflow.
    peak = max(probabilities)
    weights = [(p / peak) ** (1.0 / temperature) for p in probabilities]
    adjusted = [w / sum(weights) for w in weights]
    return candidate_weights(adjusted, top_k=top_k, top_p=top_p)


def pick(choices, u):
    # Inverse CDF: same uniform u for each rule.
    cumulative = 0.0
    for index, probability in choices:
        cumulative += probability
        if u < cumulative:
            return index
    return choices[-1][0]  # floating-point rounding fallback


def greedy_rule(probabilities):
    return [(max(range(len(probabilities)), key=probabilities.__getitem__), 1.0)]


def dense(choices):
    values = dict(choices)
    return [values.get(i, 0.0) for i in range(len(TOKENS))]

rules = {
    "Greedy": greedy_rule(PROBS),
    "T=0.5": apply_rule(PROBS, temperature=0.5),
    "T=1": apply_rule(PROBS),
    "T=2": apply_rule(PROBS, temperature=2.0),
    "Top-k=2": apply_rule(PROBS, top_k=2),
    "Top-p=0.90": apply_rule(PROBS, top_p=0.90),
}
print("Context A | probabilities AFTER the decoding rule")
print(f"{'Rule':<13}" + "".join(f"{t:>9}" for t in TOKENS))
for name, choices in rules.items():
    print(f"{name:<13}" + "".join(f"{p:>9.3f}" for p in dense(choices)))
    assert isclose(sum(dense(choices)), 1.0)

**확인:** $T=0.5$와 $T=2$에서 mat의 확률을 강의 17장과 비교한다. top-p는 왜 mat, floor, couch를 남기는가?  
다음 셀에서는 문맥 A와 B를 비교한다. $k$와 $p$의 역할을 후보 개수와 남겨진 확률로 구분해서 읽는다.

In [ ]:
for ctx, probabilities in [("A", PROBS), ("B", CONCENTRATED)]:
    print(f"Context {ctx}")
    for rule, choices in [
        ("top-k=2", apply_rule(probabilities, top_k=2)),
        ("top-p=.90", apply_rule(probabilities, top_p=.90)),
    ]:
        retained = [TOKENS[i] for i, _ in choices]
        original_mass = sum(probabilities[i] for i, _ in choices)
        final = {TOKENS[i]: round(q, 3) for i, q in choices}
        print(f"  {rule:<10} {len(retained)} candidates, original mass={original_mass:.3f}, final={final}")

### 2-2. 200회 추출 — 이론적 확률과 관측 비율 구분하기 (슬라이드 19)

각 규칙은 **같은 문맥에서 같은 난수 200개**로 다음 토큰 하나씩을 독립적으로 선택한다. 한 행의 결과는 한 토큰 실험 200회의 집계이다.

- `support`: 이론상 확률이 양수인 후보 수.
- `seen`: 실제 뽑힌 서로 다른 토큰 수.
- `valid_seen`: 실제 뽑힌 서로 다른 허용 토큰 수.
- `invalid_theory`: 금지 토큰들의 확률 합.
- `invalid_count`, `invalid_rate`: 실제 금지 토큰 횟수와 비율.

같은 200개 난수를 사용해도 분포가 달라지면 추출 결과는 달라진다. 표의 두 비율은 유한 표본에서는 정확히 같을 필요가 없다.

In [ ]:
def evaluate(choices, uniforms):
    counts = Counter(TOKENS[pick(choices, u)] for u in uniforms)
    forbidden_mass = sum(q for i, q in choices if TOKENS[i] not in ALLOWED)
    forbidden_count = sum(n for token, n in counts.items() if token not in ALLOWED)
    return {
        "support": len(choices),
        "seen": len(counts),
        "valid_seen": len(set(counts) & ALLOWED),
        "invalid_theory": forbidden_mass,
        "invalid_count": forbidden_count,
        "invalid_rate": forbidden_count / len(uniforms),
        "counts": counts,
    }

results = {name: evaluate(choices, draws) for name, choices in rules.items()}
print(f"{'Rule':<13}{'Support':>9}{'Seen':>7}{'Valid seen':>12}{'Theory':>10}{'Observed':>11}{'Invalid n':>12}")
for name, r in results.items():
    print(f"{name:<13}{r['support']:>9}{r['seen']:>7}{r['valid_seen']:>12}"
          f"{r['invalid_theory']:>10.1%}{r['invalid_rate']:>11.1%}{r['invalid_count']:>12}")
    assert sum(r['counts'].values()) == N_DRAWS

print("\nCounts by token (same 200 uniforms)")
print(f"{'Rule':<13}" + "".join(f"{t:>9}" for t in TOKENS))
for name, r in results.items():
    print(f"{name:<13}" + "".join(f"{r['counts'][t]:>9}" for t in TOKENS))

# Match the exact in-class comparison, not a random new seed.
assert results['T=1']['invalid_count'] == 11
assert results['T=2']['invalid_count'] == 39

**읽어 보기:** 두 온도에서 실제 등장한 종류가 모두 5개라고 해도, 분포가 같다고 결론 낼 수 있을까? `invalid_count`와 각 토큰의 횟수를 비교하자. 200회 관측에서 금지 토큰이 0회라는 사실과 이론 확률이 0이라는 사실도 구분한다.

### 2-3. Top-p는 항상 허용된 결과를 보장할까? (슬라이드 20)

Context C에서는 모델이 `roof`에 0.62, `mat`에 0.05를 부여한다. 허용 목록은 바꾸지 않는다.

**비교 순서:** A에서 top-p → C에서 동일 top-p → C에서 허용 목록을 먼저 적용·재정규화하고 top-p. 허용 목록은 확률이 높은 순서와 다른 **과업의 외부 제약**이다.

In [ ]:
def allowed_top_p(probabilities, threshold):
    allowed_ids = [i for i, token in enumerate(TOKENS) if token in ALLOWED]
    allowed_probs = [probabilities[i] for i in allowed_ids]
    filtered = apply_rule(allowed_probs, top_p=threshold)
    return [(allowed_ids[j], q) for j, q in filtered]

context_rules = {
    "A: top-p=.90": apply_rule(PROBS, top_p=.90),
    "C: top-p=.90": apply_rule(CONTEXT_C, top_p=.90),
    "C: allowlist then top-p": allowed_top_p(CONTEXT_C, .90),
}
context_results = {}
for name, choices in context_rules.items():
    r = context_results[name] = evaluate(choices, draws)
    retained = ", ".join(TOKENS[i] for i, _ in choices)
    print(f"{name:<26} {retained:<22} | theory {r['invalid_theory']:>5.1%} | "
          f"observed {r['invalid_count']:>3}/{N_DRAWS} ({r['invalid_rate']:.1%})")
    print("  final distribution:", {TOKENS[i]: round(q, 3) for i, q in choices})

assert isclose(context_results['C: top-p=.90']['invalid_theory'], .62/.92)
assert context_results['C: top-p=.90']['invalid_count'] == 141
assert context_results['C: allowlist then top-p']['invalid_count'] == 0
assert context_results['C: allowlist then top-p']['invalid_theory'] == 0

## §3. Explain — 목적에 맞는 규칙 선택하기 (5분)

**슬라이드 19–21.** 숫자만 나열하지 말고 무엇을 원하는지부터 정의한다.

| Task (과업) | 선택할 규칙 | 근거 (이론 확률 + 관측 횟수/종류) |
|---|---|---|
| A. 같은 문맥에서 유효 라벨 하나를 일관되게 반환 | … | … |
| B. 같은 문맥에서 가능한 한 여러 유효 라벨 생성 | … | … |
| C. 문맥이 바뀌어도 금지 라벨을 절대 출력하지 않음 | … | … |

**질문 1.** 온도를 1에서 2로 올린 경우, 200회 동안 관측 종류 수는 같았는데도 어떤 성질이 바뀌었는가? 이론 확률과 실제 빈도로 설명하라.  
**질문 2.** A에서 잘 작동한 top-p가 C에서는 왜 실패하는가? $0.62/0.92$의 의미를 설명하라.  
**질문 3.** 허용 목록을 적용하면 이 한 토큰 실험에서 금지어를 막을 수 있다. 그렇다면 긴 답변의 사실성까지 보장되는가?

**내 답변:** …

> 참고: 이 실험에서 Task A/B는 제품 요구사항이고 Context A/B/C는 모델의 확률표다. 위 표에서는 원래 Context A에서 Task A/B를 먼저 판단한 뒤, Context C에서 규칙을 다시 검증한다.

## §4. Implement — Temperature softmax 한 줄 구현하기 (6분)

**슬라이드 17, 25.** logit $z$에서 온도 $T>0$를 이용해 확률을 계산한다. 수치 안정성을 위해 모든 logit에서 최대값 $m$을 뺀다.

$$p_i(T)=\frac{\exp((z_i-m)/T)}{\sum_j\exp((z_j-m)/T)}.$$

공통 상수 $m$을 빼도 정규화 후 확률은 달라지지 않는다. 아래 함수의 마지막 한 줄만 수정한 뒤 검증 셀을 실행한다.

In [ ]:
def temperature_softmax(logits, temperature):
    if temperature <= 0:
        raise ValueError("Temperature must be positive.")
    peak = max(logits)
    weights = [exp((score - peak) / temperature) for score in logits]
    return None  # TODO: weights의 합을 이용해 각 원소를 정규화한 리스트를 반환한다.

In [ ]:
if temperature_softmax(LOGITS, 1.0) is None:
    print("§4 TODO를 완성한 뒤 위 함수와 이 셀을 다시 실행하세요.")
else:
    for t in (0.5, 1.0, 2.0):
        answer = temperature_softmax(LOGITS, t)
        reference = dense(apply_rule(PROBS, temperature=t))
        assert len(answer) == len(TOKENS)
        assert all(p >= 0 for p in answer) and isclose(sum(answer), 1.0)
        assert all(isclose(a, b, rel_tol=1e-10, abs_tol=1e-12) for a, b in zip(answer, reference))
        shifted = temperature_softmax([z + 500 for z in LOGITS], t)
        assert all(isclose(a, b, rel_tol=1e-10, abs_tol=1e-12) for a, b in zip(answer, shifted))
        print(f"T={t}: mat={answer[0]:.3f}, floor={answer[1]:.3f}, sum={sum(answer):.6f}")
    print("PASS: probability normalization, equivalence to §2, and logit-shift invariance.")

**본실습 끝.** §1의 예측과 §3의 판단이 일치했는지 확인하자.  
**Key idea:** 모델이 계산한 확률, 그 확률에서 토큰을 선택하는 규칙, 과업에 맞는지 평가하는 기준은 서로 다른 문제이다.

---
## §5. Optional — AR, tokenization, training loss 복습
*본실습 이후 또는 자율 복습용; 강의 v16의 추가 설명과 연결한다.*

### 5-1. AR의 chain rule과 next-token cross-entropy (슬라이드 4, 13–14)

`I love cats`의 정답 토큰 확률이 0.50, 0.40, 0.25라면 경로 확률은 이 세 수의 곱이다. 이 곱의 음의 로그를 각 위치로 분해하면 next-token NLL의 합이 된다. 평균을 내면 token-level cross-entropy다. 아래에는 EOS를 포함하지 않았다(세 토큰 prefix의 확률).

In [ ]:
from math import prod
sequence = [("BOS", "I", .50), ("I", "love", .40), ("I love", "cats", .25)]
path_probability = prod(p for _, _, p in sequence)
losses = [-log(p) for _, _, p in sequence]
for (prefix, token, p), loss in zip(sequence, losses):
    print(f"{prefix:<10} -> {token:<5}  p={p:.2f}  -ln(p)={loss:.3f}")
print(f"Prefix probability = {path_probability:.4f}")
print(f"Sum NLL = {sum(losses):.6f}; mean CE = {sum(losses)/len(losses):.6f}")
assert isclose(path_probability, 0.05)
assert isclose(-log(path_probability), sum(losses))
print("Same training idea: DDPM predicts added noise (MSE); AR predicts true next token (CE).")

### 5-2. Token IDs, embeddings and causal mask (슬라이드 8–12)

슬라이드 9의 3차원 벡터와 ID는 실제 tokenizer가 아니라 설명용 사전이다. 여기서는 ID를 학습된 embedding의 행을 선택하는 식별자로 사용한다. 같은 토큰을 두 번 lookup하면 같은 벡터를 얻는다. 이후 실제 Transformer에서는 위치 정보와 문맥 의존 연산으로 hidden state가 달라질 수 있다. 이 예제는 **lookup과 mask만** 보여 주며 attention 가중치나 `bank`의 의미를 실제로 학습하는 모델은 아니다.

In [ ]:
toy_ids = {"I": 42, "love": 315, "cats": 827}
toy_embedding = {42: (.2, -.4, .8), 315: (.7, .3, -.2), 827: (-.1, .9, .4)}
text = "I love cats"
toy_tokens = text.split()  # simplified: one word per token
ids = [toy_ids[token] for token in toy_tokens]
print("Toy tokens:", toy_tokens)
print("Toy IDs:", ids)
print("Lookup vectors:", [toy_embedding[i] for i in ids])

# Teacher forcing: current input predicts its following observed token.
inputs = ["The", "cat", "sat"]
targets = ["cat", "sat", "EOS"]  # assume the sequence ends here
print("\nCausal attention visibility (1=allowed, 0=blocked)")
print(f"{'Input':<8}" + "".join(f"{token:>7}" for token in inputs) + "  Target")
for i, (token, target) in enumerate(zip(inputs, targets)):
    mask = [int(j <= i) for j in range(len(inputs))]
    print(f"{token:<8}" + "".join(f"{v:>7}" for v in mask) + f"  {target}")
print("During training the true prefixes are known; during AR generation they must be built step by step.")

## §6. Optional — Greedy vs. Beam Search (슬라이드 16)

이제부터는 한 위치를 200번 뽑는 실험과 달리 **두 토큰을 연속 생성**한다. 첫 토큰이 달라지면 다음 확률도 달라진다. 강의에서처럼 같은 길이의 경로를 누적 확률로 비교한다. width 2는 첫 단계에서 `mat`과 `floor`를 남긴다. 다음 단계에서 각 가지의 상위 두 연장만 표시해도 전체 상위 두 경로를 결정할 수 있다. 아래 네 행은 표시한 후보들이며 모든 전체 문장 경로의 최적해를 보장하지 않는다.

In [ ]:
first_step = dict(zip(TOKENS, PROBS))
# The top two extensions per retained parent (slide 16).
extensions = {"mat": {"today": .30, "again": .24},
              "floor": {"quietly": .95, "today": .05}}
first_beam = sorted(first_step, key=first_step.get, reverse=True)[:2]
paths = [(first_step[a] * p_b, a, b)
         for a in first_beam for b, p_b in extensions[a].items()]
ranked_paths = sorted(paths, key=lambda row: row[0], reverse=True)
greedy_a = first_beam[0]
greedy_b = max(extensions[greedy_a], key=extensions[greedy_a].get)
print(f"Greedy path: {greedy_a} {greedy_b}")
print("Beam width=2: expand both parents, pool, then keep top 2")
for rank, (score, a, b) in enumerate(ranked_paths, 1):
    print(f"{rank}: {a:>5} {b:<8} {first_step[a]:.2f} × {extensions[a][b]:.2f} = {score:.4f} "
          f"{'KEEP' if rank <= 2 else 'prune'}")
assert ranked_paths[0][1:] == ("floor", "quietly")
assert isclose(ranked_paths[0][0], .1995)
assert isclose(ranked_paths[1][0], .1860)

**생각해 볼 질문:** Beam width를 늘리면 계산은 어떻게 달라질까? 가장 확률 높은 두 토큰 경로가 항상 과업에 맞거나 사실인 답이라는 보장은 있는가?

## §7. Optional — Temperature, top-p를 직접 조합해 보기

모델과 난수는 그대로 두고 $T$와 top-p만 바꿔 Context A/C에서 결과가 어떻게 달라지는지 비교한다. 이는 새 실험이며 슬라이드 19–20의 고정 결과와 혼동하지 않는다. 허용 목록을 적용한 결과는 §2-3에서 확인했다. 아래 셀의 값을 바꾸고 다시 실행할 수 있다.

In [ ]:
EXPERIMENT_T = 1.0
EXPERIMENT_P = 0.90
for ctx in ("A", "C"):
    choices = apply_rule(CONTEXTS[ctx], temperature=EXPERIMENT_T, top_p=EXPERIMENT_P)
    r = evaluate(choices, draws)
    print(f"Context {ctx}: T={EXPERIMENT_T}, top-p={EXPERIMENT_P}")
    print("  candidates:", {TOKENS[i]: round(p, 4) for i, p in choices})
    print(f"  allowed types observed={r['valid_seen']}; forbidden theory={r['invalid_theory']:.1%}; "
          f"observed={r['invalid_count']}/{N_DRAWS}")

## §8. Optional — 실제 pretrained language model (수업 후)

[DistilGPT2 model card](https://huggingface.co/distilbert/distilgpt2). 이 섹션만 인터넷과 `torch`, `transformers`, 모델 다운로드가 필요하다. 설치가 필요하면 별도의 셀에서 `%pip install torch transformers`를 실행한다. 다운로드를 원할 때만 아래 `RUN_PRETRAINED_DEMO`를 `True`로 바꾼다.

앞 슬라이드의 숫자는 가상 예시였지만, 여기서는 실제 tokenizer의 조각과 ID, 다음 토큰 확률을 확인한다. 같은 모델과 prompt, seed에서 온도만 바꾼 두 continuation을 비교한다. 모델은 instruction-tuned chat assistant가 아니며, 두 출력만으로 모델 성능이나 사실성을 평가할 수 없다.

In [ ]:
RUN_PRETRAINED_DEMO = False
if RUN_PRETRAINED_DEMO:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_id = "distilbert/distilgpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id).eval()
    prompt = "The cat sat on the"
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"][0]
    print("Tokens and IDs:", list(zip(tokenizer.convert_ids_to_tokens(input_ids.tolist()), input_ids.tolist())))
    with torch.inference_mode():
        probs = model(**inputs).logits[0, -1].softmax(dim=-1)
        values, ids = torch.topk(probs, 5)
        print("Top five real next-token probabilities:",
              [(repr(tokenizer.decode([i])), round(float(p), 4)) for p, i in zip(values.tolist(), ids.tolist())])
        for temperature in (0.5, 1.5):
            torch.manual_seed(SEED)
            output = model.generate(**inputs, max_new_tokens=24, do_sample=True,
                                    temperature=temperature, top_k=0, top_p=1.0,
                                    pad_token_id=tokenizer.eos_token_id)
            continuation = output[0, len(input_ids):]
            print(f"T={temperature}: {tokenizer.decode(continuation, skip_special_tokens=True)!r}")
else:
    print("Optional pretrained demo skipped; core lab needs no download.")

### References and scope
- Week 13 lecture: `Week13_LLMs_and_AR_at_Scale_v16(1).pptx` (31 slides), especially slides 4, 8–20, 24–25.
- Holtzman et al., *The Curious Case of Neural Text Degeneration*, ICLR 2020. [Paper](https://arxiv.org/abs/1904.09751)
- Vaswani et al., *Attention Is All You Need*, NeurIPS 2017. [Paper](https://arxiv.org/abs/1706.03762)
- Source lab: `W13_decoding_v3.ipynb`. The five-token probabilities, seed 13, 200 draws and observed counts are preserved.